# Interactive Results Explorer
## The Familiar Stranger Problem: Packet-Disjoint vs. Identity-Disjoint Evaluation (ESDA 2026)

This notebook allows interactive inspection and visualization of the experimental results generated across the 20 evaluation episodes on the 60-device LoRa RFFI benchmark dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("Libraries loaded successfully.")

### 1. Overall Statistics & Device-Level Bootstrap Confidence Intervals
Evaluated over the 60 physical devices (resampling physical transmitters rather than pseudo-replicated episodes).

In [ ]:
df_stats = pd.read_csv('../results/tables/stats_summary.csv')
display(df_stats)

### 2. Main Episode Summary (20 Episodes)
Comparison of Condition A (Familiar Strangers, FAR_A) vs. Condition B (True Strangers, FAR_B).

In [ ]:
df_episodes = pd.read_csv('../results/tables/episode_summary.csv')
display(df_episodes.head(10))

pos_gap = (df_episodes['DeltaFAR'] > 0).mean() * 100
print(f"Episodes with positive Familiarity Gap (DeltaFAR > 0): {pos_gap:.1f}%")
print(f"Mean FAR_A (Familiar): {df_episodes['FAR_A'].mean():.4f}")
print(f"Mean FAR_B (True Strangers): {df_episodes['FAR_B'].mean():.4f}")
print(f"Mean DeltaFAR: {df_episodes['DeltaFAR'].mean():.4f}")

### 3. Device-Matched Paired Design (Confound Elimination)
Evaluates the exact same physical devices on identical test packets at full calibration (tau_full) vs. held-out calibration (tau_j).

In [ ]:
df_matched = pd.read_csv('../results/tables/matched_device_evaluation.csv')
display(df_matched[['far_familiar', 'far_unfamiliar', 'delta_far_matched']].describe())

dev_means = df_matched.groupby('device_id')['delta_far_matched'].mean().values
print(f"Device-Matched Mean DeltaFAR: {np.mean(dev_means):.4f}")

rng = np.random.default_rng(42)
boots = [np.mean(rng.choice(dev_means, size=len(dev_means), replace=True)) for _ in range(2000)]
ci_low, ci_high = np.percentile(boots, [2.5, 97.5])
print(f"95% Device Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}] (Strictly Positive)")

### 4. Proxy-Count Sweep (k = 0, 1, 3, 5, 10, 15)
Investigates the scaling of outlier exposure and the acute single-proxy (k=1) distortion.

In [ ]:
df_sweep = pd.read_csv('../results/sweep/sweep_results.csv')
sweep_summary = df_sweep.groupby('k')[['FAR_A', 'FAR_B', 'FRR', 'DeltaFAR']].mean()
display(sweep_summary)

### 5. Identity-Robust Threshold (tau_robust) Trade-Off
Evaluating the operational availability cost in battery-powered AMR networks.

In [ ]:
df_tradeoff = pd.read_csv('../results/tables/threshold_tradeoff.csv')
print("Mean Youden vs Robust Thresholds:")
print(df_tradeoff[['FAR_true_youden', 'FAR_true_robust', 'FRR_youden', 'FRR_robust']].mean())

### 6. Display Key Publication Figures

In [ ]:
print("Figure 4: Empirical Decision Score Distributions")
display(Image(filename='../results/figures/fig4_score_distributions.png'))

print("Figure 7: Proxy-Count Scaling Sweep")
display(Image(filename='../results/figures/fig7_sweep_plot.png'))

print("Figure 9: Security-Availability Trade-Off")
display(Image(filename='../results/figures/fig9_tradeoff.png'))